# RetinaNet — Detectron2

In [ ]:
import json
import math
import os
from pathlib import Path

import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

DATA_ROOT = Path(os.environ.get("FIRE_COCO_ROOT", "data/processed/coco")).resolve()
OUTPUT_DIR = Path("artifacts/checkpoints/retinanet_r50_fpn").resolve()
SEED = 42

In [ ]:
for split in ("train", "valid", "test"):
    split_dir = DATA_ROOT / split
    annotation = split_dir / "_annotations.coco.json"
    if not annotation.is_file():
        raise FileNotFoundError(annotation)
    coco = json.loads(annotation.read_text(encoding="utf-8"))
    categories = coco.get("categories", [])
    if len(categories) != 1 or categories[0].get("name") != "fire":
        raise ValueError(f"{annotation} phải chứa đúng một category 'fire'")
    name = f"forest_fire_{split}"
    if name not in DatasetCatalog.list():
        register_coco_instances(name, {}, str(annotation), str(split_dir))

In [ ]:
class FireTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(
            dataset_name,
            output_dir=output_folder or str(Path(cfg.OUTPUT_DIR) / "validation"),
        )

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_50_FPN_3x.yaml")
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.MODEL.RETINANET.NUM_CLASSES = 1
cfg.DATASETS.TRAIN = ("forest_fire_train",)
cfg.DATASETS.TEST = ("forest_fire_valid",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.INPUT.MIN_SIZE_TRAIN = (640,)
cfg.INPUT.MAX_SIZE_TRAIN = 640
cfg.INPUT.MIN_SIZE_TEST = 640
cfg.INPUT.MAX_SIZE_TEST = 640
cfg.SEED = SEED
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.0025
iters_per_epoch = math.ceil(len(DatasetCatalog.get("forest_fire_train")) / cfg.SOLVER.IMS_PER_BATCH)
cfg.SOLVER.MAX_ITER = iters_per_epoch * 25
cfg.SOLVER.STEPS = (int(cfg.SOLVER.MAX_ITER * 0.6), int(cfg.SOLVER.MAX_ITER * 0.85))
cfg.SOLVER.CHECKPOINT_PERIOD = iters_per_epoch
cfg.TEST.EVAL_PERIOD = iters_per_epoch
cfg.OUTPUT_DIR = str(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
trainer = FireTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
cfg_eval = cfg.clone()
cfg_eval.MODEL.WEIGHTS = str(OUTPUT_DIR / "model_final.pth")
cfg_eval.MODEL.RETINANET.SCORE_THRESH_TEST = 0.001
cfg_eval.DATASETS.TEST = ("forest_fire_test",)
predictor = DefaultPredictor(cfg_eval)
evaluator = COCOEvaluator("forest_fire_test", output_dir=str(OUTPUT_DIR / "test"))
loader = build_detection_test_loader(cfg_eval, "forest_fire_test")
test_metrics = inference_on_dataset(predictor.model, loader, evaluator)
test_metrics